# PSD PREDICTION ANALYSIS

In [4]:
import matplotlib.pyplot as plt  # type: ignore
import numpy as np  # type: ignore
import os
import pandas as pd  # type: ignore
from typing import Optional, cast, List, Union

import src.constants as cn
from src.psd_prediction_file_iterator import PSDPredictionFileIterator
from src.psd_analyzer import PSDAnalyzer
from src.cumulative_segment_curve import CumulativeSegmentCurve
from src.piecewise_system_discovery import PiecewiseSystemDiscovery
from src.biomodels_iterator import BiomodelsIterator
from src.timecourse_iterator import TimecourseIterator
from src.model import Model

# Data analysis

In [ ]:
def fileFilter(dct) -> bool:
    if not cn.COL_MAX_FRACTIONAL_REDUCTION in dct:
        return False
    if dct[cn.COL_MAX_FRACTIONAL_REDUCTION] !=  1e-5:
        pass
        #return False
    if dct[cn.COL_REPEAT] > 1:
        pass
        #return False
    return True



In [ ]:
def dataframeFilter(df, dct) -> pd.DataFrame:
    new_df = df.copy()
    return new_df[ new_df [cn.COL_AGGREGATION_TYPE] == "model"]

In [ ]:
df = PSDPredictionFileIterator.query()
#df = df [ df[cn.COL_SYSTEM_ID] == "BIOMD0000000297"]
df = df.reset_index()
df[[cn.COL_SYSTEM_ID, cn.COL_P50, cn.COL_AUC]]

In [ ]:
plt.scatter(df[cn.COL_AUC].values, df[cn.COL_P50].values)

In [ ]:
#df.loc[df.groupby('A')['B'].idxmin(), ['A', 'B', 'C']]
df1 = df[df[cn.COL_P50] > 0.8]
mask = df1[cn.COL_AUC].isna()
df1 = df1[~mask]
dff = df1.loc[df1.groupby(cn.COL_SYSTEM_ID)[cn.COL_AUC].idxmin(),[cn.COL_SYSTEM_ID,
            cn.COL_NUM_CHANGEPOINT, cn.COL_AUC, cn.COL_P50, cn.COL_FILENAME, cn.COL_CHANGEPOINTS]]
dff

# AUC and Model Accuracy

AUC indicates that there are 3 types of models
* Those that are well approximated by a single linear system
* Those that require a modest to significant number of piecewise approximations
* Those that require a very dense series of piecewise approximations

# Making Best Models

In [7]:
for min_accuracy in [0.8, 0.9]:
    for accuracy_stat in ["min", "p50", "max"]:
        df = PSDAnalyzer.makeBestPSDModelDF(min_accuracy=min_accuracy, accuracy_stat=accuracy_stat)
#df2 = PSDPredictionAnalyzer.makeBestPSDModelDF(min_accuracy=0.9, accuracy_stat="max")

Output file already exists at /Users/jlheller/home/Technical/repos/biomodels_linearity/data/psd_prediction_analyzer_bestmodel__minaccuracy_0.8__accuracystat_min.csv. Loading it instead of generating a new one.
Output file already exists at /Users/jlheller/home/Technical/repos/biomodels_linearity/data/psd_prediction_analyzer_bestmodel__minaccuracy_0.8__accuracystat_p50.csv. Loading it instead of generating a new one.
Output file already exists at /Users/jlheller/home/Technical/repos/biomodels_linearity/data/psd_prediction_analyzer_bestmodel__minaccuracy_0.8__accuracystat_max.csv. Loading it instead of generating a new one.
Output file already exists at /Users/jlheller/home/Technical/repos/biomodels_linearity/data/psd_prediction_analyzer_bestmodel__minaccuracy_0.9__accuracystat_min.csv. Loading it instead of generating a new one.
Output file already exists at /Users/jlheller/home/Technical/repos/biomodels_linearity/data/psd_prediction_analyzer_bestmodel__minaccuracy_0.9__accuracystat_p50

In [5]:
df = PSDAnalyzer.makeBestPSDModelDF(min_accuracy=0.8, accuracy_stat="max")

Output file already exists at /Users/jlheller/home/Technical/repos/biomodels_linearity/data/psd_prediction_analyzer_bestmodel__minaccuracy_0.8__accuracystat_max.csv. Loading it instead of generating a new one.


In [6]:
df

,level_0,index,system_id,aggregation_type,mean,min,max,count,invalid_count,p05,...,changepoints,max_changepoint,max_fractional_reduction,coefficient_threshold,num_changepoint,is_changepoint_removal,num_species,boundaries,auc,filename
0,0,0,BIOMD0000000005,model,0.134616,0.000000,0.807696,6,0,0.000000,...,[],5000,1.000000e-02,0.001,0,True,6,"[0, 99999]",0.000000,piecewise_predictions__numpoint_100000__thresh...
1,1,7,BIOMD0000000010,model,0.312271,0.000000,0.907829,8,0,0.000000,...,"[20, 40, 60, 80, 100, 120, 140, 160, 180, 200,...",5000,1.000000e-03,0.001,820,True,8,"[0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 2...",0.984104,piecewise_predictions__numpoint_100000__thresh...
2,2,16,BIOMD0000000042,model,0.065483,0.000000,0.880652,15,0,0.000000,...,[],5000,1.000000e-02,0.001,0,True,15,"[0, 99999]",0.000000,piecewise_predictions__numpoint_100000__thresh...
3,3,32,BIOMD0000000234,model,0.406161,0.000000,0.812323,2,0,0.040616,...,"[np.int64(1660), np.int64(1680), np.int64(2400...",5000,1.000000e-03,0.001,61,True,2,"[0, 1660, 1680, 2400, 2420, 2500, 2519, 4999, ...",0.630221,piecewise_predictions__numpoint_100000__thresh...
4,4,35,BIOMD0000000278,model,0.999944,0.999918,0.999978,3,0,0.999920,...,[],5000,1.000000e-02,0.001,0,True,3,"[0, 99999]",0.000000,piecewise_predictions__numpoint_100000__thresh...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,413,5008,BIOMD0000001056,model,0.752956,0.533978,0.966535,3,0,0.556416,...,"[np.int64(20), np.int64(40), np.int64(60), np....",5000,1.000000e-03,0.001,100,True,3,"[0, 20, 40, 60, 80, 100, 120, 140, 220, 240, 2...",0.651549,piecewise_predictions__numpoint_100000__thresh...
414,414,4935,BIOMD0000001057,model,0.498612,0.000000,0.997223,2,0,0.049861,...,"[20, 40, 60, 80, 100, 120, 140, 160, 280, 300,...",5000,1.000000e-03,0.001,15,True,2,"[0, 20, 40, 60, 80, 100, 120, 140, 160, 280, 3...",0.049286,piecewise_predictions__numpoint_100000__thresh...
415,415,5008,BIOMD0000001058,model,0.853689,0.401841,0.994270,10,0,0.497719,...,"[np.int64(20), np.int64(40), np.int64(60), np....",5000,1.000000e-04,0.001,687,True,10,"[0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 2...",0.971053,piecewise_predictions__numpoint_100000__thresh...
416,416,5017,BIOMD0000001059,model,0.250000,0.000000,1.000000,8,2,0.000000,...,[],5000,1.000000e-02,0.001,0,True,10,"[0, 99999]",0.000000,piecewise_predictions__numpoint_100000__thresh...


# AUC CDF

In [ ]:
PSDPredictionAnalyzer.makeAUCCDFPlots(min_accuracy=0.8)